In [ ]:
# ============================================================
# 0. Imports & Setup
# ============================================================

import os
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain.chat_models import init_chat_model

from transformers import pipeline
import torch
from scipy.signal import resample

import gradio as gr
from dotenv import load_dotenv
from difflib import SequenceMatcher
import uuid

# --- Load .env from project root ---
env_path = Path("..") / ".env"
print("Looking for .env at:", env_path.resolve())

load_dotenv(dotenv_path=env_path)
print("Loaded API key?", os.getenv("OPENAI_API_KEY") is not None)

# ============================================================
# 1. PDF Loading (from ../data/pdf)
# ============================================================

def process_pdf(pdf_directory: str):
    """
    Recursively load all PDFs from a directory into LangChain Document objects.
    """
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_dir.resolve()}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages from {pdf_file.name}")
        except Exception as e:
            print(f"Failed to load {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Load all PDFs from ../data/pdf
all_pdf_documents = process_pdf("../data/pdf")

# ============================================================
# 2. Chunking with RecursiveCharacterTextSplitter
# ============================================================

def split_documents(documents, chunk_size: int = 1000, chunk_overlap: int = 200):
    """Split documents into chunks for embedding."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\nExample chunk:")
        print("Content:", split_docs[0].page_content[:200], "...")
        print("Metadata:", split_docs[0].metadata)
    return split_docs


chunks = split_documents(all_pdf_documents)

# ============================================================
# 3. Embedding Manager (legal-BERT)
# ============================================================

class EmbeddingManager:
    """Handles document embedding generation using sentence-transformers."""

    def __init__(self, model_name: str = "nlpaueb/legal-bert-base-uncased"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load sentence-transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            dim = self.model.get_sentence_embedding_dimension()
            print(f"Model loaded successfully. Embedding dimension: {dim}")
        except Exception as e:
            print(f"Failed to load embedding model: {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if self.model is None:
            raise RuntimeError("Embedding model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print("Embedding shape:", embeddings.shape)
        return embeddings

    def get_embedding_dimension(self) -> int:
        if self.model is None:
            raise RuntimeError("Embedding model not loaded.")
        return self.model.get_sentence_embedding_dimension()


embedding_manager = EmbeddingManager()

# ============================================================
# 4. VectorStore (ChromaDB) – ALWAYS rebuilt on start
# ============================================================

class Vectorstore:
    """Manages document embeddings in a ChromaDB vector store."""

    def __init__(
        self,
        collection_name: str = "pdf_documents_legalbert",
        persist_directory: str = "../data/vector_store_legalbert",
        reset_collection: bool = True,   # <— always rebuild on script start
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store(reset_collection=reset_collection)

    def _initialize_store(self, reset_collection: bool):
        """Initialize the ChromaDB persistent store."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            print("Using persist directory:", self.persist_directory)

            self.client = chromadb.PersistentClient(path=self.persist_directory)

            if reset_collection:
                # Delete old collection if exists, so we always reflect ../data/pdf
                try:
                    self.client.delete_collection(self.collection_name)
                    print(f"Deleted existing collection: {self.collection_name}")
                except Exception:
                    print(f"No existing collection named {self.collection_name} to delete.")

            self.collection = self.client.get_or_create_collection(
                self.collection_name,
                metadata={"description": "PDF document embeddings for RAG (legal-BERT)"},
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection_count()}")
        except Exception as e:
            print("Error initializing vector store:", e)
            raise

    def collection_count(self) -> int:
        if self.collection is None:
            return 0
        return self.collection.count()

    def add_documents(self, documents, embeddings: np.ndarray, batch_size: int = 1000):
        """
        Add documents and embeddings to ChromaDB in safe batches.

        Args:
            documents: list of LangChain Document objects
            embeddings: np.ndarray of shape (n_docs, emb_dim)
            batch_size: max documents per Chroma add() call
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings.")

        n_docs = len(documents)
        print(f"Adding {n_docs} documents to vector store (batch_size={batch_size})...")
        print("Embedding dim:", len(embeddings[0]))

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # send to Chroma in batches (to avoid max-batch error)
        for start in range(0, n_docs, batch_size):
            end = min(start + batch_size, n_docs)
            print(f"  → Adding batch {start}:{end} ({end - start} docs)")
            try:
                self.collection.add(
                    ids=ids[start:end],
                    embeddings=embeddings_list[start:end],
                    metadatas=metadatas[start:end],
                    documents=documents_text[start:end],
                )
            except Exception as e:
                print(f"Error adding batch {start}:{end} to vector store:", e)
                raise

        print(f"✅ Finished adding {n_docs} documents.")
        print(f"Total documents in collection: {self.collection.count()}")


# instantiate AND rebuild the vector store from ../data/pdf
vectorstore = Vectorstore(reset_collection=True)

print("Vector store is being rebuilt – computing embeddings and adding documents...")
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embedding(texts)
vectorstore.add_documents(chunks, embeddings, batch_size=1000)

# ============================================================
# 5. RAG Retriever (improved, invoice-aware)
# ============================================================

class RAGRetriever:
    """Improved retriever with fuzzy search + keyword/filename boost."""

    def __init__(self, vectorstore: Vectorstore, embedding_manager: EmbeddingManager):
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def _fuzzy_score(self, a: str, b: str) -> float:
        """Boost short queries using fuzzy matching on the first 300 chars."""
        return SequenceMatcher(None, a.lower(), b.lower()).ratio()

    def _filename_boost(self, meta: Dict[str, Any], query: str) -> float:
        name = (meta.get("source_file") or "").lower()
        q = query.lower()
        score = 0.0

        # Strong boost for invoice / aws keywords in filename
        if "invoice" in name:
            score += 0.20
        if "aws" in name or "amazon" in name:
            score += 0.20

        # Light boost if query itself contains aws/invoice
        if "invoice" in q:
            score += 0.10
        if "aws" in q or "amazon" in q:
            score += 0.10

        return score

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        print(f"\n🔍 RAG retrieve() query: {query!r}, top_k={top_k}")

        # 1) Embed query
        query_emb = self.embedding_manager.generate_embedding([query])[0]

        # 2) Query Chroma
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_emb.tolist()],
                n_results=top_k,
            )
        except Exception as e:
            print("❌ Error querying Chroma:", e)
            return []

        retrieved_docs: List[Dict[str, Any]] = []

        if not results or not results.get("documents") or not results["documents"][0]:
            print("❌ No documents returned from Chroma.")
            return []

        docs = results["documents"][0]
        metas = results["metadatas"][0]
        dists = results["distances"][0]
        ids = results["ids"][0]

        for i, (doc_id, content, meta, dist) in enumerate(
            zip(ids, docs, metas, dists)
        ):
            cosine_like = 1 - dist           # Chroma returns distance
            fuzzy = self._fuzzy_score(query, content[:300])

            kw_content = content.lower()
            keywords = ["aws", "amazon", "invoice", "billing", "account"]
            kw_hits = sum(1 for k in keywords if k in kw_content)
            keyword_boost = kw_hits * 0.05   # text keyword boost

            filename_boost = self._filename_boost(meta, query)

            final_score = (
                0.55 * cosine_like
                + 0.20 * fuzzy
                + 0.10 * keyword_boost
                + 0.15 * filename_boost
            )

            print(
                f"  → cand {i+1}: id={doc_id}, "
                f"cos={cosine_like:.3f}, fuzzy={fuzzy:.3f}, "
                f"txt_kw={keyword_boost:.3f}, file_boost={filename_boost:.3f}, "
                f"final={final_score:.3f}, source={meta.get('source_file')}"
            )

            retrieved_docs.append(
                {
                    "id": doc_id,
                    "content": content,
                    "metadata": meta,
                    "similarity_score": final_score,
                    "distance": dist,
                    "rank": i + 1,
                }
            )

        # sort by final_score descending and keep top_k
        retrieved_docs.sort(key=lambda x: x["similarity_score"], reverse=True)
        print("\nTop reranked docs:")
        for d in retrieved_docs[:top_k]:
            print(
                f" score={d['similarity_score']:.3f}, "
                f"file={d['metadata'].get('source_file')}, len={d['metadata'].get('content_length')}"
            )
        return retrieved_docs[:top_k]


rag_retriever = RAGRetriever(vectorstore, embedding_manager)

# Quick sanity test (text-only RAG, no voice yet)
test_results = rag_retriever.retrieve("aws invoice", top_k=3)
print("Sample retrieved docs:", len(test_results))
for d in test_results:
    print(d["similarity_score"], d["metadata"].get("source_file"))

# ============================================================
# 6. Initialize Chat Model (OpenAI via LangChain)
# ============================================================

model = init_chat_model("gpt-4o", temperature=0.1, max_tokens=1024)

# ============================================================
# 7. Advanced RAG helper with fallback
# ============================================================

def rag_advanced(
    query: str,
    retriever: RAGRetriever,
    llm,
    top_k: int = 5,
    return_context: bool = False,
) -> Dict[str, Any]:
    """
    RAG pipeline:
    - Try to retrieve docs.
    - If none, fall back to answering with the LLM without context.
    """

    retrieved = retriever.retrieve(query, top_k=top_k)

    # Fallback: no docs → answer directly with GPT (no RAG)
    if not retrieved:
        print("⚠️ No docs retrieved – answering from LLM only.")
        direct_prompt = (
            "You are a helpful assistant. Answer the following question clearly:\n\n"
            f"{query}\n\nAnswer:"
        )
        resp = llm.invoke(direct_prompt)
        return {
            "answer": resp.content,
            "source": [],
            "confidence": 0.0,
            "context": "" if return_context else None,
        }

    # Normal RAG path
    context = "\n\n".join(d["content"] for d in retrieved)

    prompt = f"""
You are an expert assistant. Use ONLY the following context to answer the question.
If the context is incomplete, you may still infer an answer, but explicitly say:
"This answer is partially inferred because the PDF context is not complete."

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
""".strip()

    resp = llm.invoke(prompt)

    result = {
        "answer": resp.content,
        "source": retrieved,
        "confidence": max(d["similarity_score"] for d in retrieved),
    }
    if return_context:
        result["context"] = context

    return result

# ============================================================
# 8. ASR (Whisper) + Language Utilities
# ============================================================

# 1) ASR – Whisper small (multilingual)
asr_device = "mps" if torch.backends.mps.is_available() else "cpu"
print("ASR device:", asr_device)

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if asr_device != "cpu" else -1,  # 0 for GPU/MPS, -1 for CPU
)

# 2) Simple translations via LLM

def translate_en_to_hi(text: str) -> str:
    prompt = f"Translate the following English text to Hindi:\n\n{text}"
    response = model.invoke(prompt)
    return response.content.strip()

def translate_hi_to_en(text: str) -> str:
    prompt = f"Translate the following Hindi text to English:\n\n{text}"
    response = model.invoke(prompt)
    return response.content.strip()

# 3) Simple script detector

def detect_language_script(text: str) -> str:
    """
    Very simple detection:
    - if Devanagari characters present -> 'hi'
    - otherwise -> 'en'
    """
    for ch in text:
        if "\u0900" <= ch <= "\u097F":  # Devanagari Unicode range
            return "hi"
    return "en"

# 4) Transcription wrapper with manual resampling
#    lang_hint: "en", "hi", or "auto" from dropdown

def transcribe_audio_to_text(audio_array: np.ndarray, sr: int, lang_hint: str = "auto") -> str:
    """
    Use Whisper to transcribe audio (Hindi or English).
    Manually resample to 16 kHz to avoid torchaudio dependency.
    """
    target_sr = 16000

    # Ensure mono
    if audio_array.ndim > 1:
        audio_array = audio_array[:, 0]

    # Resample if needed
    if sr != target_sr:
        duration = audio_array.shape[0] / sr
        new_length = int(duration * target_sr)
        audio_array = resample(audio_array, new_length)
        sr = target_sr

    generate_kwargs = {}
    if lang_hint == "en":
        generate_kwargs = {"language": "en", "task": "transcribe"}
    elif lang_hint == "hi":
        generate_kwargs = {"language": "hi", "task": "transcribe"}
    # if "auto" → let Whisper auto-detect

    asr_inputs = {"array": audio_array, "sampling_rate": sr}
    asr_call_kwargs = {"return_timestamps": False}
    if generate_kwargs:
        asr_call_kwargs["generate_kwargs"] = generate_kwargs

    out = asr(asr_inputs, **asr_call_kwargs)
    text = out.get("text", "").strip()
    print("ASR text:", text if text else "(empty)")
    return text

# ============================================================
# 9. TTS Stub (you can wire real GPT TTS later)
# ============================================================

def synthesize_tts_gpt(text: str, output_path: str = "tts_output.wav") -> str:
    """
    TEMP: TTS stub.
    Implement real GPT TTS or another TTS here.
    For now, return None so Gradio knows there is no audio.
    """
    print("TTS is not implemented yet. Skipping audio generation.")
    return None

# ============================================================
# 10. RAG + Language Handling (text IO)
# ============================================================

def rag_answer_with_lang(
    user_text: str,
    retriever: RAGRetriever,
    llm_model,
    input_lang: str = "auto",   # 'auto', 'en', 'hi'
    output_lang: str = "en",    # 'en', 'hi'
    do_tts: bool = True,
) -> Dict[str, Any]:
    text = user_text.strip()
    if not text:
        return {
            "query_detected_lang": None,
            "query_used_for_rag": "",
            "answer_en": "",
            "answer_final": "",
            "audio_path": None,
            "rag_sources": [],
        }

    # 1) Detect language
    if input_lang == "auto":
        detected = detect_language_script(text)
    else:
        detected = input_lang

    print("Detected input language:", detected)

    # 2) Translate Hindi query -> English for RAG
    if detected == "hi":
        query_en = translate_hi_to_en(text)
        print("HI → EN query:", query_en)
    else:
        query_en = text

    # 3) RAG
    rag_result = rag_advanced(
        query_en,
        retriever,
        llm_model,
        top_k=3,
        return_context=True,
    )

    answer_en = rag_result["answer"]
    sources = rag_result["source"]

    # 4) Output language + optional TTS
    audio_path = None
    if output_lang == "hi":
        answer_final = translate_en_to_hi(answer_en)
        print("EN → HI answer:", answer_final)
        if do_tts:
            audio_path = synthesize_tts_gpt(answer_final)
    else:
        answer_final = answer_en

    return {
        "query_detected_lang": detected,
        "query_used_for_rag": query_en,
        "answer_en": answer_en,
        "answer_final": answer_final,
        "audio_path": audio_path,
        "rag_sources": sources,
    }

# ============================================================
# 11. Gradio App: Voice + Text Chatbot UI
# ============================================================

def gradio_rag_voice(audio, input_lang_choice, output_lang_choice, history):
    """
    audio: (sr, np.ndarray) from gr.Audio(type='numpy')
    history: chat history for Chatbot (list of (user, assistant) tuples)
    """
    if history is None:
        history = []

    if audio is None:
        debug = (
            "Audio is None → frontend didn't send anything.\n"
            "Check:\n"
            "1) Browser mic permission is ALLOWED for 127.0.0.1\n"
            "2) You see a waveform in the Gradio audio component before clicking the button\n"
            "3) Try Chrome instead of Safari/Firefox\n"
        )
        history = history + [("🔴 No audio detected", "Please record again.")]
        return history, None, debug

    sr, wav = audio
    print("Received audio from Gradio. sr:", sr, "shape:", getattr(wav, "shape", None))

    if wav.ndim > 1:
        wav = wav[:, 0]  # mono

    wav = wav.astype("float32")

    # Step 1: ASR (pass dropdown hint)
    spoken_text = transcribe_audio_to_text(wav, sr, lang_hint=input_lang_choice)
    if not spoken_text:
        debug = "ASR returned empty text – maybe audio is too quiet or too short."
        history = history + [("🎤 (silence?)", "No speech detected, please try again.")]
        return history, None, debug

    # Step 2: RAG + language pipeline
    result = rag_answer_with_lang(
        spoken_text,
        rag_retriever,
        model,
        input_lang=input_lang_choice,
        output_lang=output_lang_choice,
        do_tts=True,   # TTS stubbed; will just return None
    )

    final_answer = result["answer_final"]
    audio_path = result["audio_path"]
    detected_lang = result["query_detected_lang"]

    debug_text = (
        f"Detected language (script): {detected_lang}\n"
        f"ASR heard: {spoken_text}\n"
        f"Query sent to RAG: {result['query_used_for_rag']}\n"
        f"Sources: {[s['metadata'].get('source_file') for s in result['rag_sources']]}\n"
    )

    # Update chat history (chatbot-style)
    history = history + [(spoken_text, final_answer)]

    return history, audio_path, debug_text


def gradio_rag_text(user_text, input_lang_choice, output_lang_choice, history):
    """
    Text fallback: lets you test RAG + LLM without audio.
    """
    if history is None:
        history = []

    text = (user_text or "").strip()
    if not text:
        history = history + [("⚪️ Empty message", "Please type a question.")]
        return history, "", "No text provided."

    result = rag_answer_with_lang(
        text,
        rag_retriever,
        model,
        input_lang=input_lang_choice,
        output_lang=output_lang_choice,
        do_tts=False,
    )

    final_answer = result["answer_final"]
    detected_lang = result["query_detected_lang"]

    debug_text = (
        f"[TEXT MODE]\n"
        f"Detected language (script): {detected_lang}\n"
        f"Query sent to RAG: {result['query_used_for_rag']}\n"
        f"Sources: {[s['metadata'].get('source_file') for s in result['rag_sources']]}\n"
    )

    history = history + [(text, final_answer)]

    # Clear the textbox by returning "" as second output
    return history, "", debug_text


with gr.Blocks() as demo:
    gr.Markdown("## Voice + Text RAG over PDFs (Hindi + English) – legal-BERT embeddings")

    with gr.Row():
        input_lang = gr.Dropdown(
            ["auto", "en", "hi"],
            value="auto",
            label="Input language detection (for ASR + RAG)",
            info="For English questions like 'AWS invoice', choose 'en' for best results.",
        )
        output_lang = gr.Dropdown(
            ["en", "hi"],
            value="en",
            label="Answer language",
            info="Choose how you want the RAG answer.",
        )

    with gr.Row():
        audio_in = gr.Audio(
            sources=["microphone"],
            type="numpy",
            label="🎤 Speak your question (click mic, talk, then stop)",
        )
        text_in = gr.Textbox(
            label="💬 Or type your question",
            placeholder="e.g., Show me the AWS invoice details",
        )

    with gr.Row():
        ask_audio_btn = gr.Button("Ask with Voice")
        ask_text_btn = gr.Button("Ask with Text")

    chatbot = gr.Chatbot(
        label="Chat History",
        type="tuples",  # explicit tuples mode
    )
    audio_out = gr.Audio(
        label="Audio Answer (Hindi only; TTS stubbed so may be empty)",
        type="filepath",
    )
    debug_box = gr.Textbox(
        label="Debug Info",
        lines=8,
    )

    # Voice flow
    ask_audio_btn.click(
        fn=gradio_rag_voice,
        inputs=[audio_in, input_lang, output_lang, chatbot],
        outputs=[chatbot, audio_out, debug_box],
    )

    # Text flow (no audio)
    ask_text_btn.click(
        fn=gradio_rag_text,
        inputs=[text_in, input_lang, output_lang, chatbot],
        outputs=[chatbot, text_in, debug_box],
    )

demo.launch()
